<a href="https://colab.research.google.com/github/hiteshraos/BIS-LAB/blob/main/1BM24CS117_ELECTRICITY_DISTRIBUTION_PARTICLE_SWARM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np


def evaluate_schedule(schedule, appliances, hourly_rates, max_capacity):
    """Calculates total electricity cost + capacity penalty for a given schedule."""
    hourly_load = np.zeros(24)

    # Calculate hourly load curve
    for i, app in enumerate(appliances):
        start_time = int(schedule[i])
        duration = app["duration"]
        power = app["power"]
        hourly_load[start_time : start_time + duration] += power

    # Energy cost calculation
    energy_cost = np.sum(hourly_load * hourly_rates)

    # Capacity limit penalty
    over_limit = np.maximum(0, hourly_load - max_capacity)
    penalty = np.sum(over_limit) * 500  # Penalty factor per kW over limit

    return energy_cost + penalty


def particle_swarm_optimization(
    appliances, hourly_rates, max_capacity, num_particles=30, max_iter=100
):
    num_appliances = len(appliances)
    max_starts = np.array([24 - app["duration"] for app in appliances])

    # Initialize positions and velocities
    positions = np.array(
        [
            [np.random.randint(0, max_starts[j] + 1) for j in range(num_appliances)]
            for _ in range(num_particles)
        ],
        dtype=float,
    )
    velocities = np.zeros((num_particles, num_appliances))

    # Personal and Global Bests
    pbest_pos = np.copy(positions)
    pbest_scores = np.array(
        [
            evaluate_schedule(p, appliances, hourly_rates, max_capacity)
            for p in positions
        ]
    )

    gbest_index = np.argmin(pbest_scores)
    gbest_pos = np.copy(pbest_pos[gbest_index])
    gbest_score = pbest_scores[gbest_index]

    # Hyperparameters
    w = 0.7  # Inertia
    c1 = 1.5  # Cognitive
    c2 = 1.5  # Social

    for _ in range(max_iter):
        for i in range(num_particles):
            r1, r2 = np.random.rand(num_appliances), np.random.rand(
                num_appliances
            )

            # Velocity update
            velocities[i] = (
                w * velocities[i]
                + c1 * r1 * (pbest_pos[i] - positions[i])
                + c2 * r2 * (gbest_pos - positions[i])
            )

            # Position update
            positions[i] = np.round(positions[i] + velocities[i])

            # Boundary constraints
            positions[i] = np.clip(positions[i], 0, max_starts)

            # Evaluate fitness
            score = evaluate_schedule(
                positions[i], appliances, hourly_rates, max_capacity
            )

            # Update Personal Best
            if score < pbest_scores[i]:
                pbest_scores[i] = score
                pbest_pos[i] = np.copy(positions[i])

                # Update Global Best
                if score < gbest_score:
                    gbest_score = score
                    gbest_pos = np.copy(positions[i])

    return gbest_pos.astype(int), gbest_score


def main():
    print("--- Smart Electricity Load Scheduling using PSO ---")

    # User Input for Number of Appliances
    n = int(input("Enter number of appliances to schedule: "))
    appliances = []

    for i in range(n):
        print(f"\nAppliance {i+1}:")
        name = input("  Name: ")
        power = float(input("  Power consumption (kW): "))
        duration = int(input("  Operational duration (hours, 1-24): "))
        appliances.append({"name": name, "power": power, "duration": duration})

    max_capacity = float(input("\nEnter maximum hourly grid capacity (kW): "))

    # 24-Hour Time-of-Use Rate Vector ($/kWh)
    # Peak pricing during hours 8-11 and 17-21
    hourly_rates = np.array(
        [
            0.08,
            0.08,
            0.08,
            0.08,
            0.08,
            0.08,  # Off-peak (00:00-06:00)
            0.12,
            0.12,  # Mid-peak (06:00-08:00)
            0.25,
            0.25,
            0.25,
            0.25,  # Peak (08:00-12:00)
      0.15,
            0.15,
            0.15,
            0.15,
            0.15,  # Mid-peak (12:00-17:00)
            0.30,
            0.30,
            0.30,
            0.30,  # Peak (17:00-21:00)
            0.10,
            0.10,
            0.10,  # Off-peak (21:00-24:00)
        ]
    )

    # Run PSO Algorithm
    best_schedule, min_cost = particle_swarm_optimization(
        appliances, hourly_rates, max_capacity
    )

    # Display Results
    print("\n================ Optimal Schedule ================")
    for i, app in enumerate(appliances):
        start = best_schedule[i]
        end = start + app["duration"]
        print(
            f"{app['name']:<15}: Start at {start:02d}:00 | End at {end:02d}:00 | Power: {app['power']} kW"
        )

    print(f"\nTotal Estimated Daily Energy Cost: ${min_cost:.2f}")


if __name__ == "__main__":
    main()

--- Smart Electricity Load Scheduling using PSO ---
Enter number of appliances to schedule: 3

Appliance 1:
  Name: EV CHARGER
  Power consumption (kW): 7.0
  Operational duration (hours, 1-24): 4

Appliance 2:
  Name: WASHING MACHINE
  Power consumption (kW): 2.0
  Operational duration (hours, 1-24): 2

Appliance 3:
  Name: WATER HEATER
  Power consumption (kW): 3.5
  Operational duration (hours, 1-24): 3

Enter maximum hourly grid capacity (kW): 8.0

================ Optimal Schedule ================
EV CHARGER     : Start at 00:00 | End at 04:00 | Power: 7.0 kW
WASHING MACHINE: Start at 04:00 | End at 06:00 | Power: 2.0 kW
WATER HEATER   : Start at 21:00 | End at 24:00 | Power: 3.5 kW

Total Estimated Daily Energy Cost: $3.61
